# OFT vs FastHotStuff — Experiment Results

Visualization of consensus protocol experiments run on a **Google Cloud Confidential VM** (TEE with AMD SEV-SNP).

- **Experiment 1**: OFT (1-chain, f+1 quorum) vs FastHotStuff (2-chain, 2f+1 quorum), no Byzantine nodes
- **Experiment 2**: FastHotStuff with f=⌊(N-1)/3⌋ Byzantine fork-attackers

In [1]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os

RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "results")

with open(os.path.join(RESULTS_DIR, "exp1", "summary.json")) as f:
    exp1 = json.load(f)

with open(os.path.join(RESULTS_DIR, "exp2", "summary.json")) as f:
    exp2 = json.load(f)

print("Loaded exp1:", exp1["experiment"])
print("Loaded exp2:", exp2["experiment"])

Loaded exp1: exp1_finalization_time_vs_N
Loaded exp2: exp2_byzantine_finalization_time_vs_N


In [2]:
# Extract exp1 data
oft = exp1["results"]["oft"]
fhs = exp1["results"]["fasthotstuff"]

oft_N = [r["N"] for r in oft]
oft_lat = [r["avg_latency_ms"] for r in oft]
oft_std = [r["std_dev_ms"] for r in oft]
oft_thr = [r["avg_throughput_bps"] for r in oft]

fhs_N = [r["N"] for r in fhs]
fhs_lat = [r["avg_latency_ms"] for r in fhs]
fhs_std = [r["std_dev_ms"] for r in fhs]
fhs_thr = [r["avg_throughput_bps"] for r in fhs]

# Extract exp2 data
byz = exp2["results"]
byz_N = [r["N"] for r in byz]
byz_lat = [r["avg_latency_ms"] for r in byz]
byz_std = [r["std_dev_ms"] for r in byz]
byz_f = [r["f_byzantine"] for r in byz]
byz_thr = [r["avg_throughput_bps"] for r in byz]

print("Data extracted successfully")

Data extracted successfully


In [3]:
# Style
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

## Experiment 1: OFT vs FastHotStuff (no attacks)

Block finalization time comparison with N ∈ {5, 10, 15, ..., 50} participants.  
OFT uses 1-chain commit with f+1 quorum; FastHotStuff uses 2-chain commit with 2f+1 quorum.

In [ ]:
fig1, ax1 = plt.subplots(figsize=(12, 6))

ax1.errorbar(oft_N, oft_lat, yerr=oft_std, fmt="o-", color="#2196F3",
             linewidth=2, markersize=6, capsize=4, capthick=1.5,
             label="OFT (1-chain, f+1 quorum)")
ax1.errorbar(fhs_N, fhs_lat, yerr=fhs_std, fmt="s--", color="#FF5722",
             linewidth=2, markersize=6, capsize=4, capthick=1.5,
             label="FastHotStuff (2-chain, 2f+1 quorum)")

ax1.set_xlabel("Number of participants (N)", fontsize=14)
ax1.set_ylabel("Average block finalization time (ms)", fontsize=14)
ax1.set_title("Experiment 1: Block Finalization Time vs N\n(no Byzantine nodes, TEE environment)", fontsize=15)
ax1.set_xticks(oft_N)
ax1.legend(fontsize=12, loc="upper left")

fig1.tight_layout()
fig1.savefig(os.path.join(RESULTS_DIR, "exp1_finalization_time.png"), dpi=150)
plt.show()

## Experiment 2: FastHotStuff with Byzantine fork-attackers

Block finalization time with f=⌊(N-1)/3⌋ Byzantine nodes using the **fork** strategy (proposing blocks with stale QC).  
Compared against the no-attack FastHotStuff baseline from Experiment 1.  
N ∈ {5, 10, 15, ..., 50}.

In [ ]:
fig2, ax2 = plt.subplots(figsize=(12, 6))

ax2.errorbar(fhs_N, fhs_lat, yerr=fhs_std, fmt="s--", color="#FF5722",
             linewidth=2, markersize=6, capsize=4, capthick=1.5,
             label="FastHotStuff (no attacks)")
ax2.errorbar(byz_N, byz_lat, yerr=byz_std, fmt="D-", color="#9C27B0",
             linewidth=2, markersize=6, capsize=4, capthick=1.5,
             label="FastHotStuff (f fork-attackers)")

ax2.set_xlabel("Number of participants (N)", fontsize=14)
ax2.set_ylabel("Average block finalization time (ms)", fontsize=14)
ax2.set_title("Experiment 2: Block Finalization Time vs N\n(with f=⌊(N-1)/3⌋ Byzantine nodes, fork attack)", fontsize=15)
ax2.set_xticks(byz_N)
ax2.legend(fontsize=12, loc="upper left")

fig2.tight_layout()
fig2.savefig(os.path.join(RESULTS_DIR, "exp2_byzantine_finalization.png"), dpi=150)
plt.show()

## Throughput comparison

Average throughput (blocks/sec) across all three scenarios.

In [ ]:
fig3, ax3 = plt.subplots(figsize=(12, 6))

ax3.plot(oft_N, oft_thr, "o-", color="#2196F3", linewidth=2, markersize=6,
         label="OFT (no attacks)")
ax3.plot(fhs_N, fhs_thr, "s--", color="#FF5722", linewidth=2, markersize=6,
         label="FastHotStuff (no attacks)")
ax3.plot(byz_N, byz_thr, "D-.", color="#9C27B0", linewidth=2, markersize=6,
         label="FastHotStuff (f fork-attackers)")

ax3.set_xlabel("Number of participants (N)", fontsize=14)
ax3.set_ylabel("Average throughput (blocks/sec)", fontsize=14)
ax3.set_title("Throughput Comparison", fontsize=15)
ax3.set_xticks(oft_N)
ax3.legend(fontsize=12)

fig3.tight_layout()
fig3.savefig(os.path.join(RESULTS_DIR, "throughput_comparison.png"), dpi=150)
plt.show()

## Summary table

In [7]:
print("=" * 80)
print("Experiment 1: OFT vs FastHotStuff (no attacks)")
print("=" * 80)
print(f"{'N':>4}  {'OFT lat (ms)':>14}  {'FHS lat (ms)':>14}  {'Speedup':>8}  {'OFT thr':>9}  {'FHS thr':>9}")
print("-" * 80)
for i in range(len(oft_N)):
    speedup = fhs_lat[i] / oft_lat[i]
    print(f"{oft_N[i]:>4}  {oft_lat[i]:>14.2f}  {fhs_lat[i]:>14.2f}  {speedup:>7.1f}x  {oft_thr[i]:>8.1f}  {fhs_thr[i]:>8.1f}")

print()
print("=" * 80)
print("Experiment 2: FastHotStuff with Byzantine fork-attackers")
print("=" * 80)
print(f"{'N':>4}  {'f':>3}  {'Byz lat (ms)':>14}  {'Normal lat':>14}  {'Slowdown':>9}  {'Byz thr':>9}")
print("-" * 80)
for i in range(len(byz_N)):
    slowdown = byz_lat[i] / fhs_lat[i]
    print(f"{byz_N[i]:>4}  {byz_f[i]:>3}  {byz_lat[i]:>14.2f}  {fhs_lat[i]:>14.2f}  {slowdown:>8.1f}x  {byz_thr[i]:>8.1f}")

Experiment 1: OFT vs FastHotStuff (no attacks)
   N    OFT lat (ms)    FHS lat (ms)   Speedup    OFT thr    FHS thr
--------------------------------------------------------------------------------
   4            1.20            3.32      2.8x     272.8     158.3
   7            1.97            7.17      3.6x     179.4      84.4
  10            2.94           11.38      3.9x     206.7      90.7
  13            3.99           17.65      4.4x     186.7     112.7
  16            4.76           24.29      5.1x     159.1     104.0

Experiment 2: FastHotStuff with Byzantine fork-attackers
   N    f    Byz lat (ms)      Normal lat   Slowdown    Byz thr
--------------------------------------------------------------------------------
   4    1          400.42            3.32     120.5x       4.8
   7    2          440.75            7.17      61.4x       3.6
  10    3          766.91           11.38      67.4x       2.5
  13    4          927.73           17.65      52.6x       1.8
  16    5    